In [1]:
import argparse
import csv
import collections
import re
import sys

In [ ]:
CODON_RE = re.compile(r"^(DNAP_seg\d+)_DNAP_(\d+)_([A-Z*])_to_([A-Z*])$")
_BASES = "TCAG"
_AAS   = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODON2AA = {a+b+c: _AAS[i] for i, (a, b, c) in
            enumerate((x, y, z) for x in _BASES for y in _BASES for z in _BASES)}
_COMP = str.maketrans("ACGT", "TGCA")

In [8]:
CODON2AA

{'TTT': 'F',
 'TTC': 'F',
 'TTA': 'L',
 'TTG': 'L',
 'TCT': 'S',
 'TCC': 'S',
 'TCA': 'S',
 'TCG': 'S',
 'TAT': 'Y',
 'TAC': 'Y',
 'TAA': '*',
 'TAG': '*',
 'TGT': 'C',
 'TGC': 'C',
 'TGA': '*',
 'TGG': 'W',
 'CTT': 'L',
 'CTC': 'L',
 'CTA': 'L',
 'CTG': 'L',
 'CCT': 'P',
 'CCC': 'P',
 'CCA': 'P',
 'CCG': 'P',
 'CAT': 'H',
 'CAC': 'H',
 'CAA': 'Q',
 'CAG': 'Q',
 'CGT': 'R',
 'CGC': 'R',
 'CGA': 'R',
 'CGG': 'R',
 'ATT': 'I',
 'ATC': 'I',
 'ATA': 'I',
 'ATG': 'M',
 'ACT': 'T',
 'ACC': 'T',
 'ACA': 'T',
 'ACG': 'T',
 'AAT': 'N',
 'AAC': 'N',
 'AAA': 'K',
 'AAG': 'K',
 'AGT': 'S',
 'AGC': 'S',
 'AGA': 'R',
 'AGG': 'R',
 'GTT': 'V',
 'GTC': 'V',
 'GTA': 'V',
 'GTG': 'V',
 'GCT': 'A',
 'GCC': 'A',
 'GCA': 'A',
 'GCG': 'A',
 'GAT': 'D',
 'GAC': 'D',
 'GAA': 'E',
 'GAG': 'E',
 'GGT': 'G',
 'GGC': 'G',
 'GGA': 'G',
 'GGG': 'G'}

In [10]:
CODON2AA.get("TTT", "X")

'F'

In [ ]:
def revcomp(s):
    return s.translate(_COMP)[::-1]

def translate(nt):
    return "".join(CODON2AA.get(nt[i:i+3], "X") for i in range(0, len(nt) - 2, 3))

def read_fasta_single(path):
    seq = []
    for line in open(path):
        if not line.startswith(">"):
            seq.append(line.strip())
    return "".join(seq).upper()

def segment_consensus(oligos):
    
    L = len(oligos[0][1])
    consensus = []
    for i in range(L):
        counts = collections.Counter(s[i] for _, s in oligos)
        consensus.append(counts.most_common(1)[0][0])
    return "".join(consensus)

def load_varints(csv_path):
    byseg = collections.defaultdict(list)

    with open(csv_path)

In [ ]:
plasmid_path = '/Users/cristian.soitu/Data/KhoaChung/mutational_scanning/pFR494_pRT300_rham_wt.fa'
csv_path = '/Users/cristian.soitu/Data/KhoaChung/mutational_scanning/DMS_segments.csv'


plasmid = read_fasta_single(plasmid_path)
w1, w0 = 1790, 3448
wt_win = plasmid[w1:w0]
win_len = len(wt_win)
wt_coding_aa = translate(revcomp(wt_win))

'GGTTAATTGGTTGCTGCAGTGGGTTGATGATACCGCTGCCTTACTGGGTGCATTAGCCAGTCTGAATGACCTGTCACGGGATAATCCGAAGTGGTCAGACTGGAAAATCAGAGGGCAGGAACTGCTGAACAGCAAAAAGTCAGATAGCACCACATAGCAGACCCGCCATAAAACGCCCTGAGAAGCCCGTGACGGGCTTTTCTTGTATTATGGGTAGTTTCCTTGCATGAATCCATAAAAGGCGCCTGTAGTGCCATTTACCCCCATTCACTGCCAGAGCCGTGAGCGCAGCGAACTGAATGTCACGAAAAAGACAGCGACTCAGGTGCCTGATGGTCGGAGACAAAAGGAATATTCAGCGATTTGCCCGAGCTTGCGAGGGTGCTACTTAAGCCTTTAGGGTTTTAAGGTCTGTTTTGTAGAGGAGCAAACAGCGTTTGCGACATCCTTTTGTAATACTGCGGAACTGACTAAAGTAGTGAGTTATACACAGGGCTGGGATCTATTCTTTTTATCTTTTTTTATTCTTTCTTTATTCTATAAATTATAACCACTTGAATATAAACAAAAAAAACACACAAAGGTCTAGCGGAATTTACAGAGGGTCTAGCAGAATTTACAAGTTTTCCAGCAAAGGTCTAGCAGAATTTACAGATACCCACAACTCAAAGGAAAAGGACTAGTAATTATCATTGACTAGCCCATCTCAATTGGTATAGTGATTAAAATCACCTAGACCAATTGAGATGTATGTCTGAATTAGTTGTTTTCAAAGCAAATGAACTAGCGATTAGTCGCTATGACTTAACGGAGCATGAAACCAAGCTAATTTTATGCTGTGTGGCACTACTCAACCCCACGATTGAAAACCCTACAAGGAAAGAACGGACGGTATCGTTCACTTATAACCAATACGCTCAGATGATGAACATCAGTAGGGAAAATGCTTATGGTGTATTAGCTAAAGCAACCAGAGAGCTGATGACGAGAACTGTGGAA